# Exercise 9 : Visualization

Run the first cell : it reads the World Bank file `world_ind_pop_data.csv` from Exercise 2 and
pivots it into `urban`, with one row per `Year` from 1960 to 2014 and one column per country :
`Brazil`, `China`, `India` and `Nigeria`. The values are the urban population as a percentage of
each country's population. Steps 7 and 8 use `tips.csv`.

Create every figure with `fig, ax = plt.subplots()` or `fig, axes = plt.subplots(rows, columns)`
and draw with `ax.` methods.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA = Path.cwd().parents[1] / 'datasets'
world = pd.read_csv(DATA / 'exercises' / 'world_ind_pop_data.csv')
countries = ['Brazil', 'China', 'India', 'Nigeria']
urban = world.loc[world['CountryName'].isin(countries)].pivot(
    index='Year', columns='CountryName', values='Urban population (% of total)'
)
urban.head()

## Step 1 : Line chart

Create `fig, ax = plt.subplots()`. Plot `urban['China']` against `urban.index` in blue and
`urban['India']` in red, both on `ax`.

Why does a line chart suit these yearly percentages? What can it not tell you about the number of
people who live in cities?

In [ ]:
fig, ax = plt.subplots()
ax.plot(urban.index, urban['China'], color='blue')
ax.plot(urban.index, urban['India'], color='red')
plt.show()

## Step 2 : Side by side

Create `fig, axes = plt.subplots(1, 2)`. Plot China in blue on `axes[0]` and India in red on
`axes[1]`. Give each panel a title with `set_title` and call `fig.tight_layout()` so that the
titles and labels fit.

Compare the y-axes of the two panels. Why do the two lines look alike?

In [ ]:
fig, axes = plt.subplots(1, 2)
axes[0].plot(urban.index, urban['China'], color='blue')
axes[0].set_title('China')
axes[1].plot(urban.index, urban['India'], color='red')
axes[1].set_title('India')
fig.tight_layout()
plt.show()

## Step 3 : Grid and style

Print `plt.style.available`. Inside `with plt.style.context('bmh'):`, create
`fig, axes = plt.subplots(2, 2)`. Plot Brazil in green on `axes[0, 0]`, China in blue on
`axes[0, 1]`, India in red on `axes[1, 0]` and Nigeria in purple on `axes[1, 1]`. Give each panel
a title and call `fig.tight_layout()`.

In [ ]:
print(plt.style.available)

with plt.style.context('bmh'):
    fig, axes = plt.subplots(2, 2)
    colors = ['green', 'blue', 'red', 'purple']
    for ax, country, color in zip(axes.flat, countries, colors):
        ax.plot(urban.index, urban[country], color=color)
        ax.set_title(country)
    fig.tight_layout()
    plt.show()

## Step 4 : Limits and saving

Plot China and India on one `ax`. Give each line a `label`, draw one solid and one dashed, and call
`ax.legend()`. Set the x-range to 1980–2014 with `ax.set_xlim()` and the y-range to 0–60 with
`ax.set_ylim()`, and label both axes. Save the figure as `xlim_and_ylim.png` with `fig.savefig()`.
Would the chart still work when printed in grayscale?

Make the same chart again, but set both ranges with one call, `ax.axis((1980, 2014, 0, 60))`, and
save it as `axis_limits.png`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(urban.index, urban['China'], label='China')
ax.plot(urban.index, urban['India'], label='India', linestyle='--')
ax.set_xlim(1980, 2014)
ax.set_ylim(0, 60)
ax.set(xlabel='Year', ylabel='Urban population (% of total)')
ax.legend()
fig.savefig('xlim_and_ylim.png')
plt.show()

fig, ax = plt.subplots()
ax.plot(urban.index, urban['China'], label='China')
ax.plot(urban.index, urban['India'], label='India', linestyle='--')
ax.axis((1980, 2014, 0, 60))
ax.set(xlabel='Year', ylabel='Urban population (% of total)')
ax.legend()
fig.savefig('axis_limits.png')
plt.show()

## Step 5 : Legend and annotation

Plot all four countries with their names as labels. Place the legend in the upper left with
`ax.legend(loc='upper left')` and label the y-axis `'Urban population (% of total)'`. Does the
legend cover a line? Try another `loc` if it does.

Find the first year in which more than half of China's population lived in cities, as `crossed`,
and China's share in that year, as `share`. A Boolean mask on `urban['China']` selects the years
from `urban.index`. In a new figure, plot the China line and mark that point with
`ax.annotate('Over 50% urban', ...)`, with `xy=(crossed, share)`, `xytext=(crossed - 25, share)`
and `arrowprops=dict(facecolor='black')`.

In [ ]:
fig, ax = plt.subplots()
for country in countries:
    ax.plot(urban.index, urban[country], label=country)
ax.legend(loc='upper left')
ax.set_ylabel('Urban population (% of total)')
plt.show()

crossed = urban.index[urban['China'] > 50][0]
share = urban.loc[crossed, 'China']

fig, ax = plt.subplots()
ax.plot(urban.index, urban['China'])
ax.annotate('Over 50% urban', xy=(crossed, share), xytext=(crossed - 25, share),
            arrowprops=dict(facecolor='black'))
plt.show()

## Step 6 : pandas plotting

Plot all four countries of `urban` with one call to `.plot(ax=ax)`. Add axis labels and a title
that states what the chart shows.

Make a second figure with `plot(subplots=True, layout=(2, 2))`, one panel per country. Which of the
two figures compares the countries better, and which shows each country's changes better?

In [ ]:
fig, ax = plt.subplots()
urban.plot(ax=ax)
ax.set(title='Brazil urbanized first; China passed India in the 1980s', xlabel='Year',
       ylabel='Urban population (% of total)')
plt.show()

axes = urban.plot(subplots=True, layout=(2, 2), sharex=True)
plt.show()

## Step 7 : Seaborn

Read `tips.csv`, with the columns `total_bill`, `tip`, `sex`, `day` and `time`. Draw the mean tip
per day with `sns.catplot`, with `kind='bar'`, `x='day'`, `y='tip'`, `hue='sex'`, `col='time'`,
`errorbar=('ci', 95)` and `seed=0`. Save the grid with `g.savefig('tips_by_time.png')`.

A bar does not show how many bills its mean rests on. Count the bills per `time`, `day` and `sex`
with `groupby(..., observed=True).size().reset_index(name='n')` and write the table to
`tips_group_sizes.csv` without the index. Do the counts add up to all 244 bills? Which bar rests on
a single bill?

In [ ]:
tips = pd.read_csv(DATA / 'tips.csv')
g = sns.catplot(data=tips, kind='bar', x='day', y='tip', hue='sex', col='time',
                errorbar=('ci', 95), seed=0)
g.savefig('tips_by_time.png')
plt.show()

sizes = tips.groupby(['time', 'day', 'sex'], observed=True).size().reset_index(name='n')
sizes.to_csv('tips_group_sizes.csv', index=False)
print(sizes)
print(sizes['n'].sum(), len(tips))

## Step 8 : Chart choice

Is a tip proportional to the bill? Answer with a chart of `tips` from this module, with a title
that states what you found, and save it as `tip_vs_bill.png`. Explain in two sentences in the
answer cell why you chose this chart and what it cannot show.

In [ ]:
fig, ax = plt.subplots()
sns.regplot(data=tips, x='total_bill', y='tip', seed=0, scatter_kws={'s': 10}, ax=ax)
ax.set(title='Tips rise with the bill, by about 11 cents per dollar',
       xlabel='Total bill ($)', ylabel='Tip ($)')
fig.savefig('tip_vs_bill.png')
plt.show()

A scatter plot with a fitted line suits two numeric variables : each bill is one point, and the
line shows about 11 cents more tip per extra dollar. The line does not pass through zero, so tips
are only roughly proportional to the bill, and the chart shows an association without its
causes.